NLTK 패키지 다운로드

In [1]:
!pip install nltk

In [2]:
import nltk

## [예제1] 구구조 구문 분석 규칙 작성
문법 설명:

- S: 문장(Sentence)
- NP: 명사구(Noun Phrase)
- VP: 동사구(Verb Phrase)
- PP: 전치사구(Prepositional Phrase)
- Det: 한정사(Determiner)
- N: 명사(Noun)
- V: 동사(Verb)
- P: 전치사(Preposition)

In [7]:
# NLTK 라이브러리에서 CFG(문맥 자유 문법)를 정의하는 기능을 사용
grammar = nltk.CFG.fromstring("""
# 문장의 시작 규칙 (S): 문장은 NP(명사구) + VP(동사구) 구조로 이루어짐
S -> NP VP

# 명사구(NP): NN(명사) + XSN(접미사) + JK(조사) 또는 NN + JK 구조
NP -> NN XSN JK | NN JK

# 동사구(VP): NP(명사구) + VP(동사구) 또는 VV(동사어간) + EP(선어말 어미) + EF(종결 어미) 구조
VP -> NP VP | VV EP EF

# 명사(NN): '아이' 또는 '케이크' 단어를 명사로 정의
NN -> '아이' | '케이크'

# 명사 파생 접미사(XSN): 복수를 나타내는 '들'
XSN -> '들'

# 조사(JK): 주격 조사 '이' 또는 목적격 조사 '를'
JK -> '이' | '를'

# 동사 어간(VV): '먹' (동사 '먹다'의 어간 부분)
VV -> '먹'

# 선어말 어미(EP): 과거 시제를 나타내는 '었'
EP -> '었'

# 종결 어미(EF): 문장을 끝맺는 '다'
EF -> '다'
""")

규칙 기반 구문 분석기 생성 및 구구조 구문 분석 수행

제시된 ChartParser 외에도 ShiftReduceParser, RecursiveDescentParser 등

다양한 구문 분석 알고리즘이 제공된다.

In [8]:
# ChartParser: NLTK에서 제공하는 구문 분석기
# 위에서 정의한 CFG(grammar)를 이용해서 문장을 파싱할 수 있는 parser 객체 생성
parser = nltk.ChartParser(grammar)

In [5]:
sent = ['아이', '들', '이', '케이크', '를', '먹', '었', '다']

In [12]:
# parser로 문장 구문 분석 수행
for tree in parser.parse(sent):
    print(tree)          # 파싱 트리 구조를 출력
    tree.pretty_print()  # 트리를 보기 좋게 출력
#    tree.draw()         # 그래픽 GUI 창으로 트리를 시각화

(S
  (NP (NN 아이) (XSN 들) (JK 이))
  (VP (NP (NN 케이크) (JK 를)) (VP (VV 먹) (EP 었) (EF 다))))
             S                     
      _______|___________           
     |                   VP        
     |            _______|___       
     NP          NP          VP    
  ___|___     ___|___     ___|___   
 NN XSN  JK  NN      JK  VV  EP  EF
 |   |   |   |       |   |   |   |  
 아이  들   이  케이크      를   먹   었   다 



## [예제2] 간단한 영어 문장을 파싱하기 위한 문맥자유문법(CFG) 정의
문법 설명:

- S: 문장(Sentence)
- NP: 명사구(Noun Phrase)
- VP: 동사구(Verb Phrase)
- PP: 전치사구(Prepositional Phrase)
- Det: 한정사(Determiner)
- N: 명사(Noun)
- V: 동사(Verb)
- P: 전치사(Preposition)

In [13]:
grammar = nltk.CFG.fromstring("""
    S -> NP VP
    NP -> Det N | N
    VP -> V NP | VP PP | V
    PP -> P NP
    Det -> 'the' | 'a'
    N -> 'cat' | 'dog' | 'man' | 'telescope'
    V -> 'saw' | 'walked'
    P -> 'with' | 'in'
""")

In [14]:
parser = nltk.ChartParser(grammar)  # 정의한 문법을 사용하여 Chart Parser 생성

In [15]:
# 파싱할 문장을 토큰화하여 리스트로 준비
sentence = "the man saw the dog with the telescope".split()

In [16]:
# parser.parse(sentence): 주어진 문장을 CFG 문법에 따라 파싱한 결과(트리들)를 반환
# list(...) : 여러 파싱 결과를 리스트로 변환하여 저장
# 즉, parses에는 문장에 대한 모든 가능한 구문 분석 트리들이 담김
parses = list(parser.parse(sentence))

In [17]:
# 파싱 결과 출력
for tree in parses:
    print(tree)
    tree.pretty_print()

(S
  (NP (Det the) (N man))
  (VP
    (VP (V saw) (NP (Det the) (N dog)))
    (PP (P with) (NP (Det the) (N telescope)))))
                 S                                
      ___________|_______                          
     |                   VP                       
     |            _______|________                 
     |           VP               PP              
     |        ___|___         ____|___             
     NP      |       NP      |        NP          
  ___|___    |    ___|___    |     ___|______      
Det      N   V  Det      N   P   Det         N    
 |       |   |   |       |   |    |          |     
the     man saw the     dog with the     telescope



# [예제3] 중국어 구문분석 예제

In [23]:
# -*- coding: utf-8 -*-
# 중국어 미니 CFG로 SVO / 把(ba) / 被(bei) 구조를 파싱하는 NLTK 예제
# 실전 전체 문법이 아니라, 데모 문장 2개를 파싱할 수 있도록 축약한 교육용 규칙입니다.

import nltk

# 1) 교육용 중국어 CFG 정의
grammar_zh = nltk.CFG.fromstring("""
# ---- 문장 규칙: 기본 SVO, 把문, 被수동문 ----
S -> NP VP PU
S -> NP BA NP VP PU
S -> NP BEI NP VP PU

# ---- 구 성분 ----
NP -> NOUN | PRON | PROPN | NOUN NOUN
VP -> V | V NP | V NP ASP | V ASP

# ---- 기능어(표지) ----
BA  -> '把'
BEI -> '被'
PU  -> '。'
ASP -> '了'

# ---- 어휘(데모 문장에 맞춤) ----
PRON -> '我' | '他'
PROPN -> '张三'
NOUN -> '学校' | '中文' | '苹果' | '书'
V -> '学习' | '吃' | '买'
ADP -> '在'

# ---- (선택) 전치사구를 문장에 포함시키고 싶다면 주석 해제 ----
# S -> NP PP VP PU
# PP -> ADP NP
""")

In [24]:
# 2) 파서(ChartParser) 생성
parser = nltk.ChartParser(grammar_zh)

# 3) 예제 문장(토큰화 완료 상태로 입력)
#    3-1) 把문: 张三 把 苹果 吃 了 。
sent1 = ['张三','把','苹果','吃','了','。']

#    3-2) 被수동문: 书 被 他 买 了 。
sent2 = ['书','被','他','买','了','。']

In [25]:
# 4) 파싱 + 출력 유틸 함수
def parse_and_show(sentence):
    """주어진 토큰 리스트를 CFG로 파싱하고 트리를 텍스트로 출력"""
    parses = list(parser.parse(sentence))      # 가능한 모든 파싱트리를 리스트화
    if not parses:
        print('[파싱 실패]', ' '.join(sentence))
        return
    print('=== 입력 ===')
    print(' '.join(sentence))
    print('=== 파싱 트리(텍스트) ===')
    for i, tree in enumerate(parses, 1):
        print(f'\n# Parse {i}')
        print(tree)            # 한 줄 트리
        print('\n[pretty_print]')
        tree.pretty_print()    # 계층형(ASCII) 트리
        # tree.draw()          # GUI 트리(로컬 환경에서만; 노트북/서버에선 비권장)

In [26]:
# 5) 데모 실행
parse_and_show(sent1)
parse_and_show(sent2)

=== 입력 ===
张三 把 苹果 吃 了 。
=== 파싱 트리(텍스트) ===

# Parse 1
(S (NP (PROPN 张三)) (BA 把) (NP (NOUN 苹果)) (VP (V 吃) (ASP 了)) (PU 。))

[pretty_print]
           S                  
   ________|________________   
  NP   |   NP       VP      | 
  |    |   |     ___|___    |  
PROPN  BA NOUN  V      ASP  PU
  |    |   |    |       |   |  
  张三   把   苹果   吃       了   。 

=== 입력 ===
书 被 他 买 了 。
=== 파싱 트리(텍스트) ===

# Parse 1
(S (NP (NOUN 书)) (BEI 被) (NP (PRON 他)) (VP (V 买) (ASP 了)) (PU 。))

[pretty_print]
          S                  
  ________|________________   
 NP   |   NP       VP      | 
 |    |   |     ___|___    |  
NOUN BEI PRON  V      ASP  PU
 |    |   |    |       |   |  
 书    被   他    买       了   。 

